In [ ]:
import pandas as pd

# File paths
bom_file   = r"D:/Tushar/main_with_subs_only.xlsx"
indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# Fixed month column
MONTH_COL = "Feb'26"          # ← confirmed by you
DAYS_IN_MONTH = 28            # Feb 2026

print("Starting calculation for Feb'26 (28 days)...\n")

# 1. Read files
print("Reading files...")
try:
    bom_df    = pd.read_excel(bom_file)
    indent_df = pd.read_excel(indent_file)
except Exception as e:
    print("Error reading files:", e)
    input("Press Enter to exit...")
    exit()

# Basic cleaning of column names
bom_df.columns    = bom_df.columns.str.strip()
indent_df.columns = indent_df.columns.str.strip()

# 2. Prepare indent data (switch → daily demand)
if MONTH_COL not in indent_df.columns:
    print(f"Column '{MONTH_COL}' not found in indent file!")
    print("Available columns:", list(indent_df.columns))
    input("Press Enter to exit...")
    exit()

# Assume switch code column is 'Part number' – change if wrong
SWITCH_COL_INDENT = 'Part number'   # ← change this if it's different (e.g. 'Switch Part', 'FG Code')

if SWITCH_COL_INDENT not in indent_df.columns:
    print(f"Switch column '{SWITCH_COL_INDENT}' not found!")
    print("Available columns:", list(indent_df.columns))
    input("Press Enter to exit...")
    exit()

indent_df = indent_df[[SWITCH_COL_INDENT, MONTH_COL]].dropna(subset=[MONTH_COL])
indent_df[MONTH_COL] = pd.to_numeric(indent_df[MONTH_COL], errors='coerce').fillna(0)

# Convert to daily demand
indent_df['Daily_Demand'] = indent_df[MONTH_COL] / DAYS_IN_MONTH

# Make switch codes clean strings
indent_df[SWITCH_COL_INDENT] = indent_df[SWITCH_COL_INDENT].astype(str).str.strip()

# Create lookup dict: switch → daily demand
daily_lookup = dict(zip(indent_df[SWITCH_COL_INDENT], indent_df['Daily_Demand']))

print(f"Loaded {len(daily_lookup)} switches with daily demand\n")

# 3. Prepare BOM
BOM_CHILD_COL = 'Main_Label'
BOM_SWITCH_COL = 'Sub_Label'
BOM_QTY_COL   = 'Sub_Count'

bom_df = bom_df[[BOM_CHILD_COL, BOM_SWITCH_COL, BOM_QTY_COL]].copy()
bom_df[BOM_CHILD_COL] = bom_df[BOM_CHILD_COL].astype(str).str.strip()
bom_df[BOM_SWITCH_COL] = bom_df[BOM_SWITCH_COL].astype(str).str.strip()
bom_df[BOM_QTY_COL]   = pd.to_numeric(bom_df[BOM_QTY_COL], errors='coerce').fillna(0)

print(f"BOM has {len(bom_df)} rows\n")

# 4. Row-by-row accumulation (to match your manual thinking)
print("Processing row by row (showing first 10 for verification)...\n")

daily_totals = {}  # child → running daily total

for idx, row in bom_df.iterrows():
    child  = row[BOM_CHILD_COL]
    switch = row[BOM_SWITCH_COL]
    qty_per = row[BOM_QTY_COL]
    
    daily_switch = daily_lookup.get(switch, 0.0)
    daily_child_from_this = daily_switch * qty_per
    
    # Accumulate
    if child in daily_totals:
        daily_totals[child] += daily_child_from_this
    else:
        daily_totals[child] = daily_child_from_this
    
    # Debug print first 10 rows
    if idx < 10:
        print(f"Row {idx+1}: Child={child:>12} | Switch={switch:>12} | Qty/Switch={qty_per:>4} | "
              f"Daily Switch={daily_switch:>8.3f} | Daily Child from this={daily_child_from_this:>8.3f} | "
              f"Running total for child={daily_totals[child]:>8.3f}")

print(f"\nProcessed {len(bom_df)} rows → found {len(daily_totals)} unique child parts\n")

# 5. Create final DataFrame
result = pd.DataFrame({
    'Child_Part': list(daily_totals.keys()),
    'Daily_Req': list(daily_totals.values()),
})
result['Two_Day_Req'] = result['Daily_Req'] * 2
result = result.sort_values('Two_Day_Req', ascending=False).round({'Daily_Req': 2, 'Two_Day_Req': 2})

print("Top 15 child parts by 2-day requirement:")
print(result.head(15))

# 6. Save
output_path = "Feb26_2_Day_Child_Requirement.xlsx"
result.to_excel(output_path, index=False)
print(f"\nSaved to: {output_path}")
print("Done.")